## 1 - Packages

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from time import time

%matplotlib inline

print("Packages imported successfully!")

Packages imported successfully!


## 2 - Load Preprocessed Data

In [32]:
# Load preprocessed data
X = joblib.load('../data/processed/X_vectorized.pkl')
Y = joblib.load('../data/processed/Y_labels.pkl')
vectorizer = joblib.load('../data/processed/vectorizer.pkl')
label_encoder = joblib.load('../data/processed/label_encoder.pkl')

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"\nSparse matrix format: {type(X)}")

X shape: (8462, 90493)
Y shape: (8462,)
Number of features: 90493
Number of samples: 8462

Sparse matrix format: <class 'scipy.sparse._csr.csr_matrix'>


## 3 - Prepare Binary Dimensions

In [33]:
# Extract binary labels for each dimension
personality_types = label_encoder.inverse_transform(Y)

dimensions = {
    'I/E': np.array([1 if pt[0] == 'I' else 0 for pt in personality_types]),
    'N/S': np.array([1 if pt[1] == 'N' else 0 for pt in personality_types]),
    'T/F': np.array([1 if pt[2] == 'T' else 0 for pt in personality_types]),
    'J/P': np.array([1 if pt[3] == 'J' else 0 for pt in personality_types])
}

# Print distributions
print("=" * 60)
print("DIMENSION DISTRIBUTIONS")
print("=" * 60)
for dim_name, labels in dimensions.items():
    pos_class = dim_name.split('/')[0]
    neg_class = dim_name.split('/')[1]
    n_pos = np.sum(labels == 1)
    n_neg = np.sum(labels == 0)
    pct_pos = 100 * n_pos / len(labels)
    print(f"{dim_name}: {pos_class}={n_pos} ({pct_pos:.1f}%), {neg_class}={n_neg} ({100-pct_pos:.1f}%)")
print("=" * 60)

DIMENSION DISTRIBUTIONS
I/E: I=6534 (77.2%), E=1928 (22.8%)
N/S: N=7284 (86.1%), S=1178 (13.9%)
T/F: T=3907 (46.2%), F=4555 (53.8%)
J/P: J=3365 (39.8%), P=5097 (60.2%)


## 4 - GridSearchCV Configuration

**Parameter Grid (OPTIMIZED FOR SPEED):**
- **C**: [0.001, 0.01, 0.1, 1.0, 10.0] - 5 values
- **penalty**: ['l1', 'l2'] - 2 regularization types
- **solver**: Automatically matched to penalty:
  - L1: 'liblinear' or 'saga'
  - L2: 'lbfgs' (default, fast)
- **class_weight**: [None, 'balanced'] - 2 options

**Total combinations per dimension:**
- L2 penalty: 5 C × 2 class_weight = 10 combinations
- L1 penalty: 5 C × 2 class_weight = 10 combinations
- **Total: 20 combinations** (much faster than SVM!)

**Cross-Validation:**
- 5-fold StratifiedKFold (preserves class proportions)
- Scoring: F1-score (better for imbalanced data)

## 4.1 - Understanding Logistic Regression Hyperparameters

### Quick Overview

**C (Regularization Strength)**: Controls the trade-off between fitting the training data and keeping the model simple. Defined as $C = 1/\lambda$ where $\lambda$ is the regularization strength. Small C (e.g., 0.001) means strong regularization → simple model that may underfit. Large C (e.g., 10.0) means weak regularization → complex model that may overfit. We test [0.001, 0.01, 0.1, 1.0, 10.0] to find the sweet spot.

**penalty (L1 vs L2)**: Determines how coefficients are penalized. L2 (Ridge) uses $\sum w_j^2$ and shrinks all coefficients proportionally, keeping all features with small weights. L1 (Lasso) uses $\sum |w_j|$ and forces many coefficients to exactly zero, performing automatic feature selection. For text data with thousands of features, L2 typically works well unless you specifically want sparse models.

**solver (Optimization Algorithm)**: The mathematical method to minimize the cost function. We use 'lbfgs' for L2 (fast quasi-Newton method, excellent for medium-large datasets) and 'liblinear' for L1 (coordinate descent, required for L1 penalty). Both are efficient for our ~8000 samples with 5000+ TF-IDF features.

**class_weight (Handle Imbalance)**: Adjusts the importance of each class in the loss function using $w_c = \frac{n_{samples}}{n_{classes} \times n_{samples\_in\_class\_c}}$. With 'balanced', minority classes get higher weights to force the model to learn both classes. Without it (None), models become "lazy" and just predict the majority class (76% accuracy but 7% specificity). This is why we test both options - balanced typically wins for imbalanced MBTI data.

**GridSearchCV Strategy**: We test 20 combinations per dimension (5 C values × 2 penalties × 2 class_weights) using 5-fold cross-validation with F1-score as the metric. All experiments use upsampling via Pipeline to prevent data leakage, ensuring balanced training within each fold.

In [34]:
# Configuration
RANDOM_STATE = 42
MAX_ITER = 1000
N_JOBS = -1  # Use all CPU cores

# Define parameter grids for different penalty types
# Note: solver must be compatible with penalty
param_grids = {
    'l2': {
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'penalty': ['l2'],
        'solver': ['lbfgs'],  # Fast solver for L2
        'class_weight': [None, 'balanced']
    },
    'l1': {
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'penalty': ['l1'],
        'solver': ['liblinear'],  # Required for L1 with small datasets
        'class_weight': [None, 'balanced']
    }
}

# Cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Configuration complete! (OPTIMIZED FOR SPEED)")
print(f"\nL2 penalty: {len(param_grids['l2']['C'])} C × {len(param_grids['l2']['class_weight'])} class_weight = {len(param_grids['l2']['C']) * len(param_grids['l2']['class_weight'])} combinations")
print(f"L1 penalty: {len(param_grids['l1']['C'])} C × {len(param_grids['l1']['class_weight'])} class_weight = {len(param_grids['l1']['C']) * len(param_grids['l1']['class_weight'])} combinations")
print(f"\nTotal per dimension: {sum(len(pg['C']) * len(pg['class_weight']) for pg in param_grids.values())} combinations")


Configuration complete! (OPTIMIZED FOR SPEED)

L2 penalty: 5 C × 2 class_weight = 10 combinations
L1 penalty: 5 C × 2 class_weight = 10 combinations

Total per dimension: 20 combinations


In [35]:
# Store all results (only with upsampling)
all_results = []

print("=" * 80)
print("NOTE: Only testing with upsampling (upsampling is essential for imbalanced data)")
print("      Without upsampling, models become 'lazy' and only predict majority class")
print("=" * 80)

NOTE: Only testing with upsampling (upsampling is essential for imbalanced data)
      Without upsampling, models become 'lazy' and only predict majority class


## 5 - Hyperparameter Tuning (With Upsampling)

**Testing with RandomOverSampler inside a Pipeline to handle class imbalance.**

- Pipeline ensures upsampling happens inside each CV fold (prevents data leakage)

Upsampling is essential for MBTI classification because:
- With it: Balanced performance with both good recall and specificity (~40-50%)
- Without it: Models achieve high accuracy (~76%) but terrible specificity (~7%) → "lazy model"

In [36]:
print("=" * 80)
print("STARTING HYPERPARAMETER TUNING (With Upsampling)")
print("=" * 80)
print("Note: Using Pipeline to prevent data leakage (upsampling happens inside each CV fold)")
print("=" * 80)

for dim_name, Y_binary in dimensions.items():
    print(f"\n{'=' * 80}")
    print(f"DIMENSION: {dim_name}")
    print("=" * 80)
    
    # Split data once (80/20 train/test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y_binary, test_size=0.2, random_state=RANDOM_STATE, stratify=Y_binary
    )
    
    print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
    
    # Test each penalty type
    for penalty_name, param_grid in param_grids.items():
        print(f"\n  Testing {penalty_name.upper()} penalty with upsampling...")
        start_time = time()
        
        # Create pipeline with upsampling + Logistic Regression
        pipeline = ImbPipeline([
            ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
            ('classifier', LogisticRegression(max_iter=MAX_ITER, random_state=RANDOM_STATE))
        ])
        
        # Adjust parameter grid for pipeline (need 'classifier__' prefix)
        pipeline_param_grid = {f'classifier__{key}': value for key, value in param_grid.items()}
        
        # GridSearchCV
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=pipeline_param_grid,
            cv=cv,
            scoring='f1',
            n_jobs=N_JOBS,
            verbose=0,
            return_train_score=True
        )
        
        grid_search.fit(X_train, y_train)
        elapsed = time() - start_time
        
        # Get best model
        best_pipeline = grid_search.best_estimator_
        best_params = grid_search.best_params_
        best_cv_score = grid_search.best_score_
        
        # Extract actual params (remove 'classifier__' prefix)
        actual_params = {key.replace('classifier__', ''): value for key, value in best_params.items()}
        
        # Evaluate on test set
        y_pred = best_pipeline.predict(X_test)
        test_accuracy = accuracy_score(y_test, y_pred)
        test_f1 = f1_score(y_test, y_pred, zero_division=0)
        test_precision = precision_score(y_test, y_pred, zero_division=0)
        test_recall = recall_score(y_test, y_pred, zero_division=0)
        
        # Store results
        result = {
            'dimension': dim_name,
            'penalty': penalty_name,
            'upsampling': True,
            'best_C': actual_params['C'],
            'best_solver': actual_params['solver'],
            'best_class_weight': actual_params['class_weight'],
            'cv_f1_score': best_cv_score,
            'test_accuracy': test_accuracy,
            'test_f1_score': test_f1,
            'test_precision': test_precision,
            'test_recall': test_recall,
            'fit_time_seconds': elapsed
        }
        all_results.append(result)
        
        print(f"    Best params: C={actual_params['C']}, class_weight={actual_params['class_weight']}")
        print(f"    CV F1: {best_cv_score:.4f}")
        print(f"    Test F1: {test_f1:.4f} | Test Acc: {test_accuracy:.4f}")
        print(f"    Time: {elapsed:.1f}s")

print(f"\n{'=' * 80}")
print("HYPERPARAMETER TUNING COMPLETE (With Upsampling)")
print("=" * 80)

STARTING HYPERPARAMETER TUNING (With Upsampling)
Note: Using Pipeline to prevent data leakage (upsampling happens inside each CV fold)

DIMENSION: I/E
Train size: 6769, Test size: 1693

  Testing L2 penalty with upsampling...
    Best params: C=10.0, class_weight=None
    CV F1: 0.8455
    Test F1: 0.8395 | Test Acc: 0.7454
    Time: 15.1s

  Testing L1 penalty with upsampling...
    Best params: C=10.0, class_weight=None
    CV F1: 0.8191
    Test F1: 0.8150 | Test Acc: 0.7165
    Time: 19.9s

DIMENSION: N/S
Train size: 6769, Test size: 1693

  Testing L2 penalty with upsampling...
    Best params: C=10.0, class_weight=None
    CV F1: 0.9068
    Test F1: 0.9126 | Test Acc: 0.8470
    Time: 10.1s

  Testing L1 penalty with upsampling...
    Best params: C=10.0, class_weight=None
    CV F1: 0.8826
    Test F1: 0.8869 | Test Acc: 0.8069
    Time: 22.6s

DIMENSION: T/F
Train size: 6769, Test size: 1693

  Testing L2 penalty with upsampling...
    Best params: C=10.0, class_weight=None
   

## 6 - Results Summary (DataFrame)

In [37]:
# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Verify all results are upsampled
print("=" * 100)
print("RESULTS SUMMARY (All with Upsampling)")
print("=" * 100)
print(f"Total configurations tested: {len(results_df)}")
print(f"All results use upsampling: {results_df['upsampling'].all()}")
print("=" * 100)

# Round numerical columns for readability
numerical_cols = ['cv_f1_score', 'test_accuracy', 'test_f1_score', 'test_precision', 'test_recall', 'fit_time_seconds']
results_df[numerical_cols] = results_df[numerical_cols].round(4)

# Sort by dimension and test F1 score
results_df = results_df.sort_values(['dimension', 'test_f1_score'], ascending=[True, False])

print("\n" + "=" * 100)
print("DETAILED RESULTS")
print("=" * 100)
display(results_df)

# Save to CSV
results_df.to_csv('logistic_regression_hyperparameter_tuning_results.csv', index=False)
print("\n Results saved to 'logistic_regression_hyperparameter_tuning_results.csv'")

RESULTS SUMMARY (All with Upsampling)
Total configurations tested: 8
All results use upsampling: True

DETAILED RESULTS


,dimension,penalty,upsampling,best_C,best_solver,best_class_weight,cv_f1_score,test_accuracy,test_f1_score,test_precision,test_recall,fit_time_seconds
0,I/E,l2,True,10.0,lbfgs,None,0.8455,0.7454,0.8395,0.8179,0.8623,15.1086
1,I/E,l1,True,10.0,liblinear,None,0.8191,0.7165,0.8150,0.8213,0.8087,19.9307
7,J/P,l1,True,1.0,liblinear,None,0.5297,0.5972,0.5465,0.4946,0.6107,14.9081
6,J/P,l2,True,1.0,lbfgs,None,0.5365,0.6208,0.5251,0.5228,0.5275,7.7590
2,N/S,l2,True,10.0,lbfgs,None,0.9068,0.8470,0.9126,0.8977,0.9279,10.1321
3,N/S,l1,True,10.0,liblinear,None,0.8826,0.8069,0.8869,0.8940,0.8799,22.5617
4,T/F,l2,True,10.0,lbfgs,None,0.7440,0.7643,0.7489,0.7373,0.7609,11.1425
5,T/F,l1,True,10.0,liblinear,None,0.7102,0.7342,0.7104,0.7150,0.7059,16.3285



 Results saved to 'logistic_regression_hyperparameter_tuning_results.csv'


## 7 - Best Model per Dimension

In [38]:
# Find best model for each dimension (by test F1 score)
best_models = results_df.loc[results_df.groupby('dimension')['test_f1_score'].idxmax()]

print("=" * 100)
print("BEST MODEL PER DIMENSION (Highest Test F1-Score)")
print("=" * 100)
display(best_models[['dimension', 'penalty', 'upsampling', 'best_C', 'best_solver', 
                      'best_class_weight', 'cv_f1_score', 'test_f1_score', 'test_accuracy', 
                      'test_recall', 'fit_time_seconds']])

print("\n" + "=" * 100)
print("KEY INSIGHTS (All models use upsampling):")
print("=" * 100)
for idx, row in best_models.iterrows():
    print(f"\n{row['dimension']}:")
    print(f"  Best penalty: {row['penalty'].upper()}")
    print(f"  C = {row['best_C']}")
    print(f"  class_weight = {row['best_class_weight']}")
    print(f"  solver = {row['best_solver']}")
    print(f"  Test F1: {row['test_f1_score']:.4f}")
    print(f"  Test Accuracy: {row['test_accuracy']:.4f}")
    print(f"  Test Recall: {row['test_recall']:.4f}")

BEST MODEL PER DIMENSION (Highest Test F1-Score)


,dimension,penalty,upsampling,best_C,best_solver,best_class_weight,cv_f1_score,test_f1_score,test_accuracy,test_recall,fit_time_seconds
0,I/E,l2,True,10.0,lbfgs,None,0.8455,0.8395,0.7454,0.8623,15.1086
7,J/P,l1,True,1.0,liblinear,None,0.5297,0.5465,0.5972,0.6107,14.9081
2,N/S,l2,True,10.0,lbfgs,None,0.9068,0.9126,0.8470,0.9279,10.1321
4,T/F,l2,True,10.0,lbfgs,None,0.7440,0.7489,0.7643,0.7609,11.1425



KEY INSIGHTS (All models use upsampling):

I/E:
  Best penalty: L2
  C = 10.0
  class_weight = None
  solver = lbfgs
  Test F1: 0.8395
  Test Accuracy: 0.7454
  Test Recall: 0.8623

J/P:
  Best penalty: L1
  C = 1.0
  class_weight = None
  solver = liblinear
  Test F1: 0.5465
  Test Accuracy: 0.5972
  Test Recall: 0.6107

N/S:
  Best penalty: L2
  C = 10.0
  class_weight = None
  solver = lbfgs
  Test F1: 0.9126
  Test Accuracy: 0.8470
  Test Recall: 0.9279

T/F:
  Best penalty: L2
  C = 10.0
  class_weight = None
  solver = lbfgs
  Test F1: 0.7489
  Test Accuracy: 0.7643
  Test Recall: 0.7609


## 8 - Export Optimal Parameters (JSON)

**This is the most important output!** Save optimal hyperparameters to JSON file for use in main analysis notebook.

All exported parameters include upsampling as they consistently achieve better F1-scores and balanced performance.

In [39]:
# Create dictionary of optimal hyperparameters for each dimension
optimal_params = {}

for idx, row in best_models.iterrows():
    optimal_params[row['dimension']] = {
        'C': float(row['best_C']),  # Convert to native Python float
        'penalty': row['penalty'],
        'solver': row['best_solver'],
        'class_weight': row['best_class_weight'],
        'max_iter': MAX_ITER,
        'random_state': RANDOM_STATE,
        'upsampling': bool(row['upsampling']),
        # Performance metrics for reference
        'expected_test_f1': float(row['test_f1_score']),
        'expected_test_accuracy': float(row['test_accuracy'])
    }

# Save to JSON
with open('../models/optimal_hyperparameters_lr.json', 'w') as f:
    json.dump(optimal_params, f, indent=2)

print("=" * 100)
print("OPTIMAL HYPERPARAMETERS EXPORTED")
print("=" * 100)
print("\nSaved to: ../models/optimal_hyperparameters_lr.json")
print("\nContents:")
print(json.dumps(optimal_params, indent=2))


OPTIMAL HYPERPARAMETERS EXPORTED

Saved to: ../models/optimal_hyperparameters_lr.json

Contents:
{
  "I/E": {
    "C": 10.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "class_weight": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 0.8395,
    "expected_test_accuracy": 0.7454
  },
  "J/P": {
    "C": 1.0,
    "penalty": "l1",
    "solver": "liblinear",
    "class_weight": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 0.5465,
    "expected_test_accuracy": 0.5972
  },
  "N/S": {
    "C": 10.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "class_weight": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected_test_f1": 0.9126,
    "expected_test_accuracy": 0.847
  },
  "T/F": {
    "C": 10.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "class_weight": null,
    "max_iter": 1000,
    "random_state": 42,
    "upsampling": true,
    "expected